In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

In [24]:
import importlib
import Model.model
import Data.Tool 

# Forcer le reload
importlib.reload(Model.model)
importlib.reload(Data.Tool)

# Si tu veux importer toutes les fonctions/classes à nouveau
from Model.model import *
from Data.Tool import *

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from Data.Tool import *
from Model.model import *
df = load_and_prepare(
        corpus_path = "../Data/train/corpus.tache1.learn.utf8",
        pkl_path    = "../Data/train/sequences_fasttext_fr.pkl",
        vector_size = 300)

📂 Chargement du corpus...
   → 57413 lignes chargées
🧹 Nettoyage du texte...
📦 Chargement des séquences depuis '../Data/train/sequences_fasttext_fr.pkl'...
   → 57413 séquences fusionnées
🗑️  Suppression des lignes vides...
   → 35 lignes supprimées | 57378 lignes restantes
🔍 Vérification cohérence tokens ↔ vecteurs...


Vérification: 100%|██████████| 57378/57378 [00:01<00:00, 37062.28it/s]


✅ Lignes finales       : 57378
✅ Balance C/M         : {'C': 49855, 'M': 7523}
✅ Erreurs restantes  : 0
🎉 Pipeline prêt pour l'entraînement !


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,     # limite vocabulaire
    ngram_range=(1,2),     # unigram + bigram
    min_df=2,              # ignore mots rares
    max_df=0.9             # ignore mots trop fréquents
)

X_tfidf = vectorizer.fit_transform(df["Texte_Nettoye"])
df["tfidf"] = list(X_tfidf.toarray())
df

,Doc_ID,Sentence_ID,Label,Texte,Cible_Chirac,Texte_Nettoye,Sequence,tfidf
0,100,1,C,"Quand je dis chers amis, il ne s'agit pas là d...",1,quand dis chers amis agit là formule diplomati...,"[[0.0745, -0.0541, 0.004, 0.0394, -0.1497, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,100,2,C,D'abord merci de cet exceptionnel accueil que ...,1,abord merci cet exceptionnel accueil congolais...,"[[-0.0682, 0.0843, 0.0088, -0.1753, -0.3258, 0...","[0.0, 0.0, 0.20680761939898232, 0.0, 0.0, 0.0,..."
2,100,3,C,C'est toujours très émouvant de venir en Afriq...,1,toujours très émouvant venir afrique car proba...,"[[-0.0493, -0.1279, 0.147, -0.1261, -0.2572, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,100,4,C,Aucun citoyen français ne peut être indifféren...,1,aucun citoyen français peut être indifférent s...,"[[0.0613, 0.0546, 0.0516, -0.009, -0.1467, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,100,5,C,"Le Congo, que naguère le <nom> qualifia de ""re...",1,congo naguère qualifia refuge liberté base dép...,"[[0.0684, 0.0858, 0.064, -0.0793, 0.0772, 0.07...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...
57373,9,389,C,Je suis heureux de le mener avec vous.,1,heureux mener,"[[0.0425, 0.1089, 0.1825, -0.159, -0.4585, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57374,9,390,C,"Vous le savez, comme vous, j'ai la passion de ...",1,savez comme passion france,"[[-0.0015, 0.0975, 0.0253, 0.0115, -0.1628, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57375,9,391,C,Je crois en son avenir.,1,crois avenir,"[[0.0425, 0.1089, 0.1825, -0.159, -0.4585, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57376,9,392,C,"Je crois en la politique, c'est-à-dire en notr...",1,crois politique dire capacité collective maîtr...,"[[0.0425, 0.1089, 0.1825, -0.159, -0.4585, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [27]:
def weighted_f1(y_vrai, y_pred):
    f1_0 = f1_score(y_vrai, y_pred, pos_label=0, zero_division=0)
    f1_1 = f1_score(y_vrai, y_pred, pos_label=1, zero_division=0)
    return (6.6 * f1_0 + 1.0 * f1_1) / 7.6

def weighted_probabilities(probs, weight=6.6):
    return (probs * weight) / (probs * weight + (1 - probs))

In [31]:

X = np.stack(df["tfidf"].values)
y = df['Cible_Chirac'].values
groups = df['Doc_ID'].values

# 2. Configuration du GroupKFold
gkf = GroupKFold(n_splits=5)

# 3. Initialisation du modèle
# On utilise class_weight pour intégrer ton facteur d'importance 6.6
# Cela dit au modèle : "Une erreur sur la classe 0 coûte 6.6 fois plus cher"
model = LogisticRegression(
    class_weight={0: 6.6, 1: 1.0}, 
    max_iter=1000, 
    solver='lbfgs'
)

scores_f1 = []

print(f"--- Baseline Logistic Regression (GroupKFold) ---")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
    # Split
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédiction (Labels)
    y_pred = model.predict(X_val)
    
    # Score F1 pour la classe 0
    score = f1_score(y_val, y_pred, pos_label=0)
    scores_f1.append(score)
    
    print(f"Fold {fold} | F1 Classe 0: {score:.4f}")

# 4. Bilan
print("\n--- PERFORMANCE GLOBALE ---")
print(f"F1-Score moyen (Classe 0) : {np.mean(scores_f1):.4f} (+/- {np.std(scores_f1):.4f})")

--- Baseline Logistic Regression (GroupKFold) ---
Fold 1 | F1 Classe 0: 0.4803
Fold 2 | F1 Classe 0: 0.5034
Fold 3 | F1 Classe 0: 0.4904
Fold 4 | F1 Classe 0: 0.5271
Fold 5 | F1 Classe 0: 0.4408

--- PERFORMANCE GLOBALE ---
F1-Score moyen (Classe 0) : 0.4884 (+/- 0.0285)


score Bi-gramme : 

--- Baseline Logistic Regression (GroupKFold) ---
Fold 1 | F1 Classe 0: 0.4803
Fold 2 | F1 Classe 0: 0.5034
Fold 3 | F1 Classe 0: 0.4904
Fold 4 | F1 Classe 0: 0.5271
Fold 5 | F1 Classe 0: 0.4408

--- PERFORMANCE GLOBALE ---
F1-Score moyen (Classe 0) : 0.4884 (+/- 0.0285)

score mono-gramme:

--- Baseline Logistic Regression (GroupKFold) ---
Fold 1 | F1 Classe 0: 0.4717
Fold 2 | F1 Classe 0: 0.4919
Fold 3 | F1 Classe 0: 0.4792
Fold 4 | F1 Classe 0: 0.5185
Fold 5 | F1 Classe 0: 0.4356

--- PERFORMANCE GLOBALE ---
F1-Score moyen (Classe 0) : 0.4794 (+/- 0.0271)

In [12]:
df = df.sort_values(['Doc_ID', 'Sentence_ID']).reset_index(drop=True)
X = np.stack(df['tfidf'].values)
y = df['Cible_Chirac'].values
groups = df['Doc_ID'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

# On isole le dataframe de validation pour y sauvegarder les probas
df_val = df.iloc[val_idx].copy()

In [10]:
# 3. ENTRAÎNEMENT DE LA RÉGRESSION LOGISTIQUE
model = LogisticRegression(class_weight={0: 6.6, 1: 1.0}, max_iter=1000, solver='lbfgs')
model.fit(X_train, y_train)

# Sauvegarde des probabilités de la classe 0 (Mitterrand) dans le df
df_val['Prob_LR'] = model.predict_proba(X_val)[:, 0]

# --- Métriques Baseline LR ---
# On convertit y_val pour que Mitterrand (0) soit la classe positive (1) pour les métriques
y_val_metric = (y_val == 0).astype(int) 
lr_probs = df_val['Prob_LR'].values
lr_preds = (lr_probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (Régression Logistique) ---")
print(f"F1-Score : {f1_score(y_val_metric, lr_preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, lr_probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, lr_probs):.4f}")

📊 --- SCORES BASELINE (Régression Logistique) ---
F1-Score : 0.4737
ROC AUC  : 0.8556
Avg Prec : 0.5426


In [ ]:
base_svm = LinearSVC(
    class_weight={0: 6.6, 1: 1.0}, 
    random_state=42, 
    max_iter=2000,
    dual=False 
)

model_svm_fast = CalibratedClassifierCV(base_svm, cv=10)

model_svm_fast.fit(X_train, y_train)

idx_class_0 = np.where(model_svm_fast.classes_ == 0)[0][0]
df_val['Prob_SVM'] = model_svm_fast.predict_proba(X_val)[:, idx_class_0]
df_val['Prob_SVM_W'] = weighted_probabilities(df_val['Prob_SVM'].values, weight=6.6)

# --- Métriques Baseline SVM Rapide ---
y_val_metric = (y_val == 0).astype(int) 
svm_probs =  df_val['Prob_SVM_W'].values
svm_preds = (svm_probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (SVM Linéaire Rapide) ---")
print(f"F1-Score : {f1_score(y_val_metric, svm_preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, svm_probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, svm_probs):.4f}")

📊 --- SCORES BASELINE (SVM Linéaire Rapide) ---
F1-Score : 0.4187
ROC AUC  : 0.8387
Avg Prec : 0.5063


In [ ]:
model = MLP(vocab_size=5000,hidden_dim=512)

trainer = TorchTrainer(
    model=model, 
    epochs=4, 
    batch_size=64, 
    lr=0.0001, 
    pos_weight=6.6, 
    device='cpu' 
)

trainer.fit(X_train, y_train, X_val=X_val, y_val=y_val)

probs_brutes = trainer.predict_proba(X_val)

df_val['Prob_MLP'] =probs_brutes 

y_val_metric = (y_val == 0).astype(int) 
probs =  df_val['Prob_MLP'].values
preds = (probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (MLP) ---")
print(f"F1-Score : {f1_score(y_val_metric, preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, probs):.4f}")

Epoch 01/4 | Train Loss: 1.5473 | Val Loss: 0.5706
Epoch 02/4 | Train Loss: 0.5920 | Val Loss: 0.5065
Epoch 03/4 | Train Loss: 0.5214 | Val Loss: 0.4676
Epoch 04/4 | Train Loss: 0.4722 | Val Loss: 0.4520
📊 --- SCORES BASELINE (MLP) ---
F1-Score : 0.2083
ROC AUC  : 0.1432
Avg Prec : 0.0653


In [ ]:
# ==========================================
# 2. FONCTION DE CRÉATION DES SÉQUENCES 3D
# ==========================================
def create_3d_sequences(df, max_len=150):
    sequences_X = []
    sequences_y = []
    vector_size = len(df['tfidf'].iloc[0])
    
    # On groupe par Doc_ID pour garder l'ordre chronologique intact
    for doc_id, group in df.groupby('Doc_ID'):
        seq_x = np.stack(group['tfidf'].values)
        seq_y = (group['Cible_Chirac'].values == 0).astype(int) 
        
        sequences_X.append(seq_x)
        sequences_y.append(seq_y)
        
    # --- PADDING ---
    X_pad = np.zeros((len(sequences_X), max_len, vector_size), dtype=np.float32)
    y_pad = np.full((len(sequences_y), max_len), -1, dtype=np.float32) # -1 pour les phrases fantômes
    
    for i, (seq_x, seq_y) in enumerate(zip(sequences_X, sequences_y)):
        length = min(len(seq_x), max_len)
        X_pad[i, :length, :] = seq_x[:length]
        y_pad[i, :length] = seq_y[:length]
        
    return X_pad, y_pad, vector_size

# On génère les tenseurs 3D proprement séparés
MAX_LEN = 500
X_train_pad, y_train_pad, VECTOR_SIZE = create_3d_sequences(df, max_len=MAX_LEN)
X_val_pad, y_val_pad, _ = create_3d_sequences(df_val, max_len=MAX_LEN)


# ==========================================
# 3. ENTRAÎNEMENT DU RNN (Bi-GRU)
# ==========================================
print(f"\n🚀 Initialisation du RNN (Vector Size: {VECTOR_SIZE})...")

# Utilisation de ton RNN défini dans model.py
rnn_model = RNN(
    vector_size=VECTOR_SIZE, 
    hidden_dim=64, 
    bi=True, 
    dropout=0.3
)

trainer = TorchTrainer(
    model=rnn_model, 
    epochs=20, 
    batch_size=16, 
    lr=0.001, 
    pos_weight=6.6, 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("\n⏳ Début de l'entraînement...")
trainer.fit(X_train_pad, y_train_pad, X_val=X_val_pad, y_val=y_val_pad)


# ==========================================
# 4. PRÉDICTION ET ÉVALUATION
# ==========================================
print("\n🔍 Remise à plat et évaluation...")

# Tenseur 3D de probabilités lissées
probs_3d = trainer.predict_proba(X_val_pad)

y_true_flat = []
probs_flat = []

for i in range(len(X_val_pad)):
    valid_len = np.sum(y_val_pad[i] != -1)
    
    y_true_flat.extend(y_val_pad[i, :valid_len])
    probs_flat.extend(probs_3d[i, :valid_len])

y_true_flat = np.array(y_true_flat)
probs_flat = np.array(probs_flat)
preds_flat = (probs_flat > 0.5).astype(int)

# Sauvegarde optionnelle dans ton df_val pour le HMM ensuite
df_val['Prob_RNN'] = probs_flat

print("📊 --- SCORES BASELINE (RNN Bi-Directionnel) ---")
print(f"F1-Score : {f1_score(y_true_flat, preds_flat):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_true_flat, probs_flat):.4f}")
print(f"Avg Prec : {average_precision_score(y_true_flat, probs_flat):.4f}")


🚀 Initialisation du RNN (Vector Size: 5000)...

⏳ Début de l'entraînement...
Epoch 01/20 | Train Loss: 1.0889 | Val Loss: 0.5292
Epoch 02/20 | Train Loss: 0.3923 | Val Loss: 0.1770
Epoch 03/20 | Train Loss: 0.1533 | Val Loss: 0.0807
Epoch 04/20 | Train Loss: 0.0893 | Val Loss: 0.0603
Epoch 05/20 | Train Loss: 0.0645 | Val Loss: 0.0408
Epoch 06/20 | Train Loss: 0.0436 | Val Loss: 0.0270
Epoch 07/20 | Train Loss: 0.0343 | Val Loss: 0.0170
Epoch 08/20 | Train Loss: 0.0263 | Val Loss: 0.0180
Epoch 09/20 | Train Loss: 0.0232 | Val Loss: 0.0194
Epoch 10/20 | Train Loss: 0.0200 | Val Loss: 0.0156
Epoch 11/20 | Train Loss: 0.0233 | Val Loss: 0.0179
Epoch 12/20 | Train Loss: 0.0206 | Val Loss: 0.0141
Epoch 13/20 | Train Loss: 0.0174 | Val Loss: 0.0173
Epoch 14/20 | Train Loss: 0.0187 | Val Loss: 0.0098
Epoch 15/20 | Train Loss: 0.0139 | Val Loss: 0.0067
Epoch 16/20 | Train Loss: 0.0101 | Val Loss: 0.0059
Epoch 17/20 | Train Loss: 0.0089 | Val Loss: 0.0076
Epoch 18/20 | Train Loss: 0.0106 | Val

In [ ]:
# ==========================================
# 4. PRÉDICTION ET ÉVALUATION CORRIGÉE
# ==========================================
print("\n🔍 Remise à plat et évaluation...")

# On récupère les probas 3D
probs_3d = trainer.predict_proba(X_val_pad)

y_true_flat = []
probs_flat = []

# On "dé-pad" pour ignorer les -1 et ne scorer que les vraies phrases
for i in range(len(X_val_pad)):
    valid_len = min(np.sum(y_val_pad[i] != -1), MAX_LEN)
    
    y_true_flat.extend(y_val_pad[i, :valid_len])
    probs_flat.extend(probs_3d[i, :valid_len])

# y_true_flat contient DÉJÀ 1 pour Mitterrand et 0 pour Autre
y_true_flat = np.array(y_true_flat)
probs_flat = np.array(probs_flat)

# Prédictions dures avec seuil à 0.5
preds_flat = (probs_flat > 0.5).astype(int)

f1_weighted= weighted_f1(y_true_flat , preds_flat )

print("📊 --- SCORES BASELINE (RNN Bi-Directionnel) ---")
print(f"F1-Score (Weighted) : {f1_weighted:.4f}")
print(f"ROC AUC             : {roc_auc_score(y_true_flat, probs_flat):.4f}")
print(f"Avg Prec (PR-AUC)   : {average_precision_score(y_true_flat, probs_flat):.4f}")


🔍 Remise à plat et évaluation...
📊 --- SCORES BASELINE (RNN Bi-Directionnel) ---
F1-Score (Weighted) : 0.9990
ROC AUC             : 1.0000
Avg Prec (PR-AUC)   : 0.9997


In [17]:
# 1. On nettoie les éventuelles lignes NaN dues à la troncature du MAX_LEN
df_val_clean = df_val.dropna(subset=['Prob_RNN']).copy()
df_val_clean = df_val_clean.rename(columns={'Prob_RNN': 'Prob_Mitterrand'})

# 2. Application du HMM par segment (Doc_ID)
A_real = compute_real_transitions(df_val_clean, target_col='Cible_Chirac', doc_col='Doc_ID')

print("📊 Matrice de Transition (A_real) :")
print(f"P(Mitterrand -> Mitterrand) : {A_real[0, 0]:.4f} | P(Mitterrand -> Autre) : {A_real[0, 1]:.4f}")
print(f"P(Autre -> Mitterrand)      : {A_real[1, 0]:.4f} | P(Autre -> Autre)      : {A_real[1, 1]:.4f}")
print("\nMatrice brute numpy :")
print(A_real)

# On utilise weight_c0=1.0 car le RNN a déjà fait le rééquilibrage de Mitterrand
df_val_clean['Prob_RNN_HMM'] = apply_hmm_proba_segmented(
    df_val_clean, 
    matrix_A=A_real, 
    weight_c0=1.0 
)

# 3. Évaluation finale
hmm_probs_final = df_val_clean['Prob_RNN_HMM'].values
hmm_preds_final = (hmm_probs_final > 0.5).astype(int)

# Vrais labels (Mitterrand = 1 pour les métriques)
y_true_metric = (df_val_clean['Cible_Chirac'] == 0).astype(int)

# Calcul du F1 pondéré
f1_hmm_weighted = f1_score(y_true_metric, hmm_preds_final, average='weighted')

def weighted_f1(y_vrai, y_pred):
    f1_0 = f1_score(y_vrai, y_pred, pos_label=0, zero_division=0)
    f1_1 = f1_score(y_vrai, y_pred, pos_label=1, zero_division=0)
    return (6.6 * f1_0 + 1.0 * f1_1) / 7.6

f1_hmm_weighted = weighted_f1(y_true_metric, hmm_preds_final)

print("🚀 --- SCORES FINAUX (Bi-LSTM + HMM) ---")
print(f"F1-Score (Weighted) : {f1_hmm_weighted:.4f}")
print(f"ROC AUC             : {roc_auc_score(y_true_metric, hmm_probs_final):.4f}")
print(f"Avg Prec (PR-AUC)   : {average_precision_score(y_true_metric, hmm_probs_final):.4f}")

📊 Matrice de Transition (A_real) :
P(Mitterrand -> Mitterrand) : 0.9452 | P(Mitterrand -> Autre) : 0.0548
P(Autre -> Mitterrand)      : 0.0074 | P(Autre -> Autre)      : 0.9926

Matrice brute numpy :
[[0.9451952  0.0548048 ]
 [0.00735294 0.99264706]]
🚀 --- SCORES FINAUX (Bi-LSTM + HMM) ---
F1-Score (Weighted) : 0.9989
ROC AUC             : 0.9999
Avg Prec (PR-AUC)   : 0.9996


In [4]:
# ==========================================
# 1. SÉCURISATION DE L'ORDRE DU TEST
# ==========================================
df_test = load_tagged_test("../Data/test/corpus.tache1.test.utf8")
df_test_context = df_test.copy()
df_test_context['is_test'] = True

# Préparation du train
df_train_context = df.copy()
df_train_context['is_test'] = False

# ==========================================
# 2. FUSION ET TRI CHRONOLOGIQUE (Pour le modèle)
# ==========================================
df_combined = pd.concat([df_train_context, df_test_context], ignore_index=True)
# Le modèle a besoin de l'ordre chronologique par débat
df_combined = df_combined.sort_values(['Doc_ID', 'Sentence_ID']).reset_index(drop=True)

📂 Chargement et parsing du jeu de test...
🧹 Nettoyage du texte (application de nettoyer_texte)...
✅ Fichier de test prêt : 27162 lignes chargées et nettoyées.


In [6]:
mask_train = ~df_combined['is_test']
mask_test = df_combined['is_test']

MAX_FEATURES = 5000
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2))

# On "fit" (apprend le vocabulaire) UNIQUEMENT sur le Train
train_texts = df_combined.loc[mask_train, 'Texte_Nettoye'].fillna("")
vectorizer.fit(train_texts)

# On transforme TOUT LE DATASET d'un coup dans une grande matrice Numpy indépendante
# (Beaucoup plus propre que d'essayer de ranger ça dans Pandas)
all_texts = df_combined['Texte_Nettoye'].fillna("")
X_tfidf_all = vectorizer.transform(all_texts).toarray() # Taille : (N_total, 5000)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)


# ==========================================
# 2. PRÉPARATION DES SÉQUENCES 3D (RNN)
# ==========================================

MAX_LEN = 500
VECTOR_SIZE = MAX_FEATURES

# On crée un index global pour savoir quelle ligne du DataFrame correspond à quelle ligne de X_tfidf_all
df_combined['global_idx'] = np.arange(len(df_combined))

# --- TRAIN ---
df_train = df_combined[mask_train].sort_values(['Doc_ID', 'Sentence_ID'])
X_train_list, y_train_list = [], []

for doc_id, group in df_train.groupby('Doc_ID'):
    # Magie : on pioche directement les vecteurs dans la matrice Numpy grâce à l'index global !
    seq_x = X_tfidf_all[group['global_idx'].values]
    
    # On met Mitterrand (Cible_Chirac == 0) en classe positive (1)
    seq_y = (group['Cible_Chirac'].values == 0).astype(int) 
    X_train_list.append(seq_x)
    y_train_list.append(seq_y)

X_train_pad = np.zeros((len(X_train_list), MAX_LEN, VECTOR_SIZE), dtype=np.float32)
y_train_pad = np.full((len(y_train_list), MAX_LEN), -1, dtype=np.float32)

for i, (seq_x, seq_y) in enumerate(zip(X_train_list, y_train_list)):
    length = min(len(seq_x), MAX_LEN)
    X_train_pad[i, :length, :] = seq_x[:length]
    y_train_pad[i, :length] = seq_y[:length]


# --- TEST ---
df_test = df_combined[mask_test].sort_values(['Doc_ID', 'Sentence_ID'])
X_test_list, indices_test_list = [], []

for doc_id, group in df_test.groupby('Doc_ID'):
    # Même chose pour le test : on pioche dans la matrice Numpy
    seq_x = X_tfidf_all[group['global_idx'].values]
    
    X_test_list.append(seq_x)
    indices_test_list.append(group['original_index'].values)

X_test_pad = np.zeros((len(X_test_list), MAX_LEN, VECTOR_SIZE), dtype=np.float32)

for i, seq_x in enumerate(X_test_list):
    length = min(len(seq_x), MAX_LEN)
    X_test_pad[i, :length, :] = seq_x[:length]


# ==========================================
# 3. ENTRAÎNEMENT DU RNN SUR TF-IDF
# ==========================================

# Modèle défini dans ton model.py
rnn_model = RNN(
    vector_size=VECTOR_SIZE, 
    hidden_dim=64, 
    bi=True, 
    dropout=0.3
)

trainer = TorchTrainer(
    model=rnn_model, 
    epochs=25, 
    batch_size=16, 
    lr=0.001, 
    pos_weight=6.6, 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

trainer.fit(X_train_pad, y_train_pad)


# ==========================================
# 4. PRÉDICTION SUR LE TEST ET SAUVEGARDE
# ==========================================
probs_3d_test = trainer.predict_proba(X_test_pad)

resultats = []

for i in range(len(X_test_pad)):
    length = min(len(X_test_list[i]), MAX_LEN)
    
    doc_probs = probs_3d_test[i, :length]
    doc_indices = indices_test_list[i][:length]
    
    for prob, orig_idx in zip(doc_probs, doc_indices):
        resultats.append({'original_index': orig_idx, 'Prob_Mitterrand': prob})

df_preds = pd.DataFrame(resultats)

df_soumission = df_test[['original_index']].copy()
df_soumission = df_soumission.merge(df_preds, on='original_index', how='left')
df_soumission['Prob_Mitterrand'] = df_soumission['Prob_Mitterrand'].fillna(0.13)
df_soumission['Prob_Mitterrand'].to_csv("submission_RNN_TFIDF.csv", index=False, header=False)

print(f"✅ Fichier 'submission_RNN_TFIDF.csv' prêt ! ({len(df_soumission)} lignes)")

Epoch 01/25 | Train Loss: 1.1269
Epoch 02/25 | Train Loss: 0.4330
Epoch 03/25 | Train Loss: 0.1657
Epoch 04/25 | Train Loss: 0.0878
Epoch 05/25 | Train Loss: 0.0596
Epoch 06/25 | Train Loss: 0.0418
Epoch 07/25 | Train Loss: 0.0304
Epoch 08/25 | Train Loss: 0.0243
Epoch 09/25 | Train Loss: 0.0217
Epoch 10/25 | Train Loss: 0.0222
Epoch 11/25 | Train Loss: 0.0213
Epoch 12/25 | Train Loss: 0.0198
Epoch 13/25 | Train Loss: 0.0169
Epoch 14/25 | Train Loss: 0.0147
Epoch 15/25 | Train Loss: 0.0160
Epoch 16/25 | Train Loss: 0.0122
Epoch 17/25 | Train Loss: 0.0103
Epoch 18/25 | Train Loss: 0.0101
Epoch 19/25 | Train Loss: 0.0101
Epoch 20/25 | Train Loss: 0.0079
Epoch 21/25 | Train Loss: 0.0094
Epoch 22/25 | Train Loss: 0.0100
Epoch 23/25 | Train Loss: 0.0076
Epoch 24/25 | Train Loss: 0.0065
Epoch 25/25 | Train Loss: 0.0059
✅ Fichier 'submission_RNN_TFIDF.csv' prêt ! (27162 lignes)


In [9]:
# ==========================================
# 1. CALCUL DE LA MATRICE DE TRANSITION SUR LE TRAIN
# ==========================================
print("🔄 Calcul de A_real sur l'entraînement...")
df_train_full = df_combined[~df_combined['is_test']].copy()

# On utilise ta fonction Tool.py
A_real_full = compute_real_transitions(
    df_train_full, 
    target_col='Cible_Chirac', 
    doc_col='Doc_ID'
)

# ==========================================
# 2. PRÉPARATION STRICTE DU JEU DE TEST
# ==========================================
# On fusionne le df_test d'origine avec les prédictions (df_preds) qu'on a générées avant
df_test_hmm = df_test.copy()
df_test_hmm = df_test_hmm.merge(df_preds, on='original_index', how='left')

# Sécurité anti-NaN
df_test_hmm['Prob_Mitterrand'] = df_test_hmm['Prob_Mitterrand'].fillna(0.13)

# Tri chronologique indispensable pour que le HMM lise dans le bon sens
df_test_hmm = df_test_hmm.sort_values(['Doc_ID', 'Sentence_ID']).reset_index(drop=True)

# ==========================================
# 3. LISSAGE HMM (SUR LE TEST UNIQUEMENT)
# ==========================================
print("⏳ Application du lissage HMM (Forward-Backward)...")

# On applique le HMM sur df_test_hmm (et SURTOUT PAS sur df_combined !)
df_test_hmm['Prob_RNN_HMM'] = apply_hmm_proba_segmented(
    df_test_hmm, 
    matrix_A=A_real_full, 
    weight_c0=1.0 
)

# ==========================================
# 4. GÉNÉRATION DU FICHIER FINAL
# ==========================================
print("💾 Génération du CSV de soumission final...")

# On remet EXACTEMENT dans l'ordre d'origine
df_soumission_finale = df_test_hmm.sort_values('original_index').copy()

# Sauvegarde propre
df_soumission_finale['Prob_RNN_HMM'].to_csv("submission_RNN_TFIDF_HMM_Final.csv", index=False, header=False)

print(f"✅ Fichier 'submission_RNN_TFIDF_HMM_Final.csv' prêt !")
print(f"📊 Nombre de lignes : {len(df_soumission_finale)}")

🔄 Calcul de A_real sur l'entraînement...
⏳ Application du lissage HMM (Forward-Backward)...
💾 Génération du CSV de soumission final...
✅ Fichier 'submission_RNN_TFIDF_HMM_Final.csv' prêt !
📊 Nombre de lignes : 27162


In [37]:
df = df.sort_values(['Doc_ID', 'Sentence_ID']).reset_index(drop=True)
df["Sequence_mean"] = df["Sequence"].apply(lambda x: np.mean(x, axis=0))
X = np.stack(df["Sequence_mean"].values)
y = df['Cible_Chirac'].values
groups = df['Doc_ID'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

# On isole le dataframe de validation pour y sauvegarder les probas
df_val = df.iloc[val_idx].copy()

In [23]:
# 3. ENTRAÎNEMENT DE LA RÉGRESSION LOGISTIQUE
model = LogisticRegression(class_weight={0: 6.6, 1: 1.0}, max_iter=1000, solver='lbfgs')
model.fit(X_train, y_train)

# Sauvegarde des probabilités de la classe 0 (Mitterrand) dans le df
df_val['Prob_LR_E'] = model.predict_proba(X_val)[:, 0]

# --- Métriques Baseline LR ---
# On convertit y_val pour que Mitterrand (0) soit la classe positive (1) pour les métriques
y_val_metric = (y_val == 0).astype(int) 
lr_probs = df_val['Prob_LR_E'].values
lr_preds = (lr_probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (Régression Logistique) ---")
print(f"F1-Score : {f1_score(y_val_metric, lr_preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, lr_probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, lr_probs):.4f}")

📊 --- SCORES BASELINE (Régression Logistique) ---
F1-Score : 0.4118
ROC AUC  : 0.8213
Avg Prec : 0.3865


In [30]:
base_svm = LinearSVC(
    class_weight={0: 6.6, 1: 1.0}, 
    random_state=42, 
    max_iter=2000,
    dual=False 
)

model_svm_fast = CalibratedClassifierCV(base_svm, cv=10)

model_svm_fast.fit(X_train, y_train)

idx_class_0 = np.where(model_svm_fast.classes_ == 0)[0][0]
df_val['Prob_SVM'] = model_svm_fast.predict_proba(X_val)[:, idx_class_0]
df_val['Prob_SVM_W'] = weighted_probabilities(df_val['Prob_SVM'].values, weight=6.6)

# --- Métriques Baseline SVM Rapide ---
y_val_metric = (y_val == 0).astype(int) 
svm_probs =  df_val['Prob_SVM_W'].values
svm_preds = (svm_probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (SVM Linéaire Rapide) ---")
print(f"F1-Score : {f1_score(y_val_metric, svm_preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, svm_probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, svm_probs):.4f}")

📊 --- SCORES BASELINE (SVM Linéaire Rapide) ---
F1-Score : 0.4171
ROC AUC  : 0.8277
Avg Prec : 0.3976


In [35]:
model = MLP(vocab_size=300,hidden_dim=512)

trainer = TorchTrainer(
    model=model, 
    epochs=30, 
    batch_size=64, 
    lr=0.0001, 
    pos_weight=6.6, 
    device='cpu' 
)

trainer.fit(X_train, y_train, X_val=X_val, y_val=y_val)

probs_brutes = trainer.predict_proba(X_val)

df_val['Prob_MLP'] =probs_brutes 

y_val_metric = (y_val == 0).astype(int) 
probs =  df_val['Prob_MLP'].values
preds = (probs > 0.5).astype(int)

print("📊 --- SCORES BASELINE (MLP) ---")
print(f"F1-Score : {f1_score(y_val_metric, preds):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val_metric, probs):.4f}")
print(f"Avg Prec : {average_precision_score(y_val_metric, probs):.4f}")

Epoch 01/30 | Train Loss: 1.1913 | Val Loss: 0.5824
Epoch 02/30 | Train Loss: 0.6375 | Val Loss: 0.5530
Epoch 03/30 | Train Loss: 0.6192 | Val Loss: 0.5408
Epoch 04/30 | Train Loss: 0.6071 | Val Loss: 0.5312
Epoch 05/30 | Train Loss: 0.5981 | Val Loss: 0.5230
Epoch 06/30 | Train Loss: 0.5898 | Val Loss: 0.5170
Epoch 07/30 | Train Loss: 0.5826 | Val Loss: 0.5144
Epoch 08/30 | Train Loss: 0.5775 | Val Loss: 0.5057
Epoch 09/30 | Train Loss: 0.5732 | Val Loss: 0.5024
Epoch 10/30 | Train Loss: 0.5686 | Val Loss: 0.4994
Epoch 11/30 | Train Loss: 0.5632 | Val Loss: 0.4946
Epoch 12/30 | Train Loss: 0.5587 | Val Loss: 0.4917
Epoch 13/30 | Train Loss: 0.5551 | Val Loss: 0.4889
Epoch 14/30 | Train Loss: 0.5550 | Val Loss: 0.4880
Epoch 15/30 | Train Loss: 0.5503 | Val Loss: 0.4849
Epoch 16/30 | Train Loss: 0.5497 | Val Loss: 0.4830
Epoch 17/30 | Train Loss: 0.5456 | Val Loss: 0.4816
Epoch 18/30 | Train Loss: 0.5441 | Val Loss: 0.4811
Epoch 19/30 | Train Loss: 0.5419 | Val Loss: 0.4808
Epoch 20/30 

In [ ]:
# ==========================================
# 2. FONCTION DE CRÉATION DES SÉQUENCES 3D
# ==========================================
def create_3d_sequences(df, max_len=150):
    sequences_X = []
    sequences_y = []
    vector_size = len(df['Sequence_mean'].iloc[0])
    
    # On groupe par Doc_ID pour garder l'ordre chronologique intact
    for doc_id, group in df.groupby('Doc_ID'):
        seq_x = np.stack(group['Sequence_mean'].values)
        seq_y = (group['Cible_Chirac'].values == 0).astype(int) 
        
        sequences_X.append(seq_x)
        sequences_y.append(seq_y)
        
    # --- PADDING ---
    X_pad = np.zeros((len(sequences_X), max_len, vector_size), dtype=np.float32)
    y_pad = np.full((len(sequences_y), max_len), -1, dtype=np.float32) # -1 pour les phrases fantômes
    
    for i, (seq_x, seq_y) in enumerate(zip(sequences_X, sequences_y)):
        length = min(len(seq_x), max_len)
        X_pad[i, :length, :] = seq_x[:length]
        y_pad[i, :length] = seq_y[:length]
        
    return X_pad, y_pad, vector_size

# On génère les tenseurs 3D proprement séparés
MAX_LEN = 500
X_train_pad, y_train_pad, VECTOR_SIZE = create_3d_sequences(df, max_len=MAX_LEN)
X_val_pad, y_val_pad, _ = create_3d_sequences(df_val, max_len=MAX_LEN)


# ==========================================
# 3. ENTRAÎNEMENT DU RNN (Bi-GRU)
# ==========================================
print(f"\n🚀 Initialisation du RNN (Vector Size: {VECTOR_SIZE})...")

# Utilisation de ton RNN défini dans model.py
rnn_model = RNN(
    vector_size=VECTOR_SIZE, 
    hidden_dim=64, 
    bi=True, 
    dropout=0.3
)

trainer = TorchTrainer(
    model=rnn_model, 
    epochs=40, 
    batch_size=16, 
    lr=0.001, 
    pos_weight=6.6, 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("\n⏳ Début de l'entraînement...")
trainer.fit(X_train_pad, y_train_pad, X_val=X_val_pad, y_val=y_val_pad)


# ==========================================
# 4. PRÉDICTION ET ÉVALUATION
# ==========================================
print("\n🔍 Remise à plat et évaluation...")

# Tenseur 3D de probabilités lissées
probs_3d = trainer.predict_proba(X_val_pad)

y_true_flat = []
probs_flat = []

for i in range(len(X_val_pad)):
    valid_len = np.sum(y_val_pad[i] != -1)
    
    y_true_flat.extend(y_val_pad[i, :valid_len])
    probs_flat.extend(probs_3d[i, :valid_len])

y_true_flat = np.array(y_true_flat)
probs_flat = np.array(probs_flat)
preds_flat = (probs_flat > 0.5).astype(int)

# Sauvegarde optionnelle dans ton df_val pour le HMM ensuite
df_val['Prob_RNN'] = probs_flat



🚀 Initialisation du RNN (Vector Size: 300)...

⏳ Début de l'entraînement...
Epoch 01/40 | Train Loss: 1.0917 | Val Loss: 0.7151
Epoch 02/40 | Train Loss: 0.8464 | Val Loss: 0.6617
Epoch 03/40 | Train Loss: 0.7908 | Val Loss: 0.6103
Epoch 04/40 | Train Loss: 0.7233 | Val Loss: 0.5439
Epoch 05/40 | Train Loss: 0.6286 | Val Loss: 0.4769
Epoch 06/40 | Train Loss: 0.6519 | Val Loss: 0.5432
Epoch 07/40 | Train Loss: 0.5874 | Val Loss: 0.5168
Epoch 08/40 | Train Loss: 0.5765 | Val Loss: 0.4210
Epoch 09/40 | Train Loss: 0.5387 | Val Loss: 0.4070
Epoch 10/40 | Train Loss: 0.5410 | Val Loss: 0.4069
Epoch 11/40 | Train Loss: 0.4892 | Val Loss: 0.3682
Epoch 12/40 | Train Loss: 0.4680 | Val Loss: 0.3786
Epoch 13/40 | Train Loss: 0.4506 | Val Loss: 0.3508
Epoch 14/40 | Train Loss: 0.4379 | Val Loss: 0.3670
Epoch 15/40 | Train Loss: 0.4171 | Val Loss: 0.3353
Epoch 16/40 | Train Loss: 0.4074 | Val Loss: 0.3469
Epoch 17/40 | Train Loss: 0.3756 | Val Loss: 0.3180
Epoch 18/40 | Train Loss: 0.3553 | Val 

In [41]:
print("📊 --- SCORES BASELINE (RNN Bi-Directionnel) ---")
print(f"F1-Score : {weighted_f1(y_true_flat, preds_flat):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_true_flat, probs_flat):.4f}")
print(f"Avg Prec : {average_precision_score(y_true_flat, probs_flat):.4f}")

📊 --- SCORES BASELINE (RNN Bi-Directionnel) ---
F1-Score : 0.9645
ROC AUC  : 0.9943
Avg Prec : 0.9593
